# negative-back — ex2: compose negative_back twice — recover grad_out through y = -(-x)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `negative-back`. Running the final beacon cell reports progress against the `Backprop: negative_back` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: negative_back` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`negative-back`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "negative-back"
DD_SUBTOPIC = "Backprop: negative_back"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `negative_back` chained twice — quick refresher

ex1 derived `negative_back = -grad_out`. The deeper facet: a graph containing TWO consecutive negations (e.g. `y = -(-x)`) must produce a leaf gradient identical to the seed.

```
u = -x         negative_back(g, _, x) = -g
y = -u         negative_back(g, _, u) = -g

g_x = negative_back(negative_back(g, _, u), _, x)
    = -(-g)
    = g
```

Each individual back fn flips sign; composing two flips recovers the original. This is the cleanest demonstration that back fns compose by ordinary function composition — there is no extra accumulation step between two single-parent ops in a chain.

### Exercise 2 — compose negative_back twice — recover grad_out through y = -(-x)

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply two-step composition of negative_back through y = -(-x) and verify the leaf gradient equals grad_out.
> Keywords: compose, double-negate, chain, identity
> ```

**KCs targeted:** `chain-rule-elementwise`, `backward-fn-signature`

Implement `negative_back(grad_out, out, x)` AND `chain_double_negate(grad_out, x_leaf)` — a tiny reverse pass over `y = -(-x_leaf)`.

Pipeline:
```
u    = -x_leaf                           # forward
y    = -u                                # forward
g_u  = negative_back(grad_out, y, u)     # = -grad_out
g_x  = negative_back(g_u, u, x_leaf)     # = -(-grad_out) = grad_out
```

The point: two single-parent back fns COMPOSE by ordinary function composition — no extra dispatcher, no parent-grads accumulation. The two sign flips cancel and the leaf grad is `grad_out`.

Tests verify:
- single-step `negative_back` for scalar, vector, matrix,
- the two-step chain returns `grad_out` exactly (no atol slack),
- the chain works for varying shapes,
- agreement with torch.autograd on `(-(-x)).sum()`.

No autograd inside your implementation.

In [ ]:
def negative_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    """dL/dx for out = -x."""
    raise NotImplementedError()


def chain_double_negate(grad_out: Tensor, x_leaf: Tensor) -> Tensor:
    """Walk reverse of y = -(-x_leaf). Returns dL/d(x_leaf) = grad_out."""
    raise NotImplementedError()


def _test_ex2():
    # --- single-step sanity ---
    x = t.tensor([1.0, -2.0, 3.0])
    g = negative_back(t.tensor([4.0, 5.0, 6.0]), -x, x)
    assert t.allclose(g, t.tensor([-4.0, -5.0, -6.0]))

    # --- two-step chain: must return grad_out EXACTLY ---
    x_leaf = t.tensor([1.0, 2.0, 3.0, 4.0])
    grad_out = t.tensor([10.0, 20.0, 30.0, 40.0])
    g_x = chain_double_negate(grad_out, x_leaf)
    assert g_x.shape == x_leaf.shape
    # Sign flips cancel: result is EXACTLY grad_out (no floating-point drift
    # because negation is exact in floats).
    assert t.equal(g_x, grad_out), f'two flips should give grad_out: {g_x}'

    # --- varying shapes ---
    rng = t.Generator().manual_seed(0)
    for shape in [(5,), (3, 4), (2, 3, 4)]:
        x_leaf = t.randn(*shape, generator=rng)
        grad_out = t.randn(*shape, generator=rng)
        g_x = chain_double_negate(grad_out, x_leaf)
        assert g_x.shape == shape, f'shape: {g_x.shape} for {shape}'
        assert t.equal(g_x, grad_out), f'identity broken at shape {shape}'

    # --- one flip is NOT identity (sanity: prove we actually compose two) ---
    x_leaf = t.tensor([1.0, 2.0])
    grad_out = t.tensor([3.0, 5.0])
    g_one = negative_back(grad_out, -x_leaf, x_leaf)
    assert not t.allclose(g_one, grad_out), 'one flip must not be identity'
    g_two = chain_double_negate(grad_out, x_leaf)
    assert t.allclose(g_two, grad_out), 'two flips must be identity'

    # --- agreement with torch.autograd ---
    x_ref = t.tensor([0.5, -1.5, 2.7], requires_grad=True)
    y = -(-x_ref)
    y.sum().backward()
    g_ours = chain_double_negate(t.ones(3), x_ref.detach())
    assert t.allclose(g_ours, x_ref.grad, atol=1e-7), (
        f'disagrees with autograd: ours={g_ours}, ref={x_ref.grad}'
    )
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def negative_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    return -grad_out


def chain_double_negate(grad_out: Tensor, x_leaf: Tensor) -> Tensor:
    u = -x_leaf
    y = -u
    g_u = negative_back(grad_out, y, u)
    g_x = negative_back(g_u, u, x_leaf)
    return g_x
```

**Why this composes exactly (no drift).** Float negation flips the sign bit — it's bit-exact. Two flips return the identical bit pattern, so `t.equal(g_x, grad_out)` (not just `allclose`) passes. Compare with `chain_exp_of_log` from the exp-back ex2, which accumulates rounding error.

**Why no accumulation between steps.** Each MiniTensor has ONE parent in this chain — `u` only feeds `y`, `x_leaf` only feeds `u`. The accumulation step in a real reverse pass only fires when a tensor has multiple downstream consumers. Single-parent chains are pure function composition.

**Why we still write the chain explicitly.** Could we hand-wave and just return `grad_out`? Yes. But this is the SHAPE every longer reverse pass has — exec each back fn, thread the output into the next as `grad_out`. Practicing on a 2-step trivial case makes the 10-step nontrivial case mechanical.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()